# Datacenter Score Analysis

This notebook ingests recent market and weather data to compute a composite datacenter siting score across U.S. grid regions.

## Environment Setup
Ensure required dependencies are available for the workflow.

## Imports and Global Configuration


In [2]:
import os
import requests
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

# Configuration
EIA_API_KEY = os.getenv('EIA_API_KEY') 
# Define the grid regions we want to score
REGION_COORDS = {
    'CAL': (36.5, -119.5), 'CAR': (35.5, -80.0), 'CENT': (38.5, -94.5),
    'FLA': (28.0, -82.0), 'MIDA': (39.0, -77.0), 'MIDW': (42.0, -89.0),
    'NE': (42.5, -72.5), 'NY': (42.9, -75.3), 'NW': (45.5, -120.5),
    'SE': (33.0, -84.0), 'SW': (36.0, -111.5), 'TEN': (36.0, -86.0),
    'TEX': (31.0, -99.0),
}

def fetch_real_demand(region):
    """
    Fetches the last 30 days of real electricity demand from EIA.
    No caching: hits the API directly.
    """
    url = 'https://api.eia.gov/v2/electricity/rto/region-data/data/'
    end = datetime.utcnow()
    start = end - timedelta(days=30)
    
    params = {
        'api_key': EIA_API_KEY,
        'data[0]': 'value',
        'facets[respondent][]': region,
        'frequency': 'hourly',
        'start': start.strftime('%Y-%m-%dT%H'),
        'end': end.strftime('%Y-%m-%dT%H'),
        'sort[0][column]': 'period',
        'sort[0][direction]': 'desc',
        'offset': 0,
        'length': 2000, 
    }
    
    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        data = r.json().get('response', {}).get('data', [])
        
        if not data: 
            return pd.DataFrame()
            
        df = pd.DataFrame(data)
        df['datetime'] = pd.to_datetime(df['period'])
        df['demand_MW'] = df['value'].astype(float)
        return df.sort_values('datetime')
        
    except Exception as e:
        print(f"⚠️ Failed to fetch demand for {region}: {e}")
        return pd.DataFrame()

In [3]:
def compute_proxies(df):
    """
    Takes real demand data and fabricates 'Proxy' values for 
    Price, Carbon, and Renewables based on load stress.
    """
    if df.empty:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    # 1. Load Stress (Real)
    # We take the mean of the most recent 168 hours (1 week)
    recent_data = df.tail(168)
    load_stress = recent_data['demand_MW'].mean()
    
    # 2. Stability Proxy (Real-ish)
    # Volatility is the standard deviation of that load
    volatility = recent_data['demand_MW'].std()

    # 3. Price Proxy (Synthetic)
    # Assumption: Price scales linearly with Demand
    # We just use the raw demand value as a 'price score' for now
    price_proxy = load_stress 

    # 4. Renewable Proxy (Synthetic)
    # Assumption: As grid gets stressed, renewables percentage drops
    # Formula: Inverse of load
    renewable_proxy = 1.0 / (1.0 + np.log(load_stress))

    # 5. Carbon Proxy (Synthetic)
    # Assumption: Carbon is the inverse of renewables
    carbon_proxy = 1.0 - renewable_proxy

    return load_stress, volatility, price_proxy, renewable_proxy, carbon_proxy

In [4]:
def run_scoring_pipeline():
    results = []
    
    print(f"🚀 Starting GridCast Scoring for {len(REGION_COORDS)} regions...")
    
    # --- A. Ingestion & Proxy Calculation ---
    for region, (lat, lon) in REGION_COORDS.items():
        print(f"  Analysing {region}...")
        
        # 1. Fetch Real Input
        df_demand = fetch_real_demand(region)
        
        # 2. Compute Proxies
        load, vol, price, renew, carbon = compute_proxies(df_demand)
        
        # 3. Fetch Temp (Optional: hardcode or use simple logic if API unwanted)
        # For this refactor, let's just use a placeholder to keep it fast
        # or use a random realistic temp if you want purely synthetic
        temp = np.random.uniform(10, 35) 
        
        results.append({
            "region": region,
            "raw_load": load,
            "raw_volatility": vol,
            "raw_price": price,
            "raw_renew": renew,
            "raw_carbon": carbon,
            "raw_temp": temp,
            "lat": lat,
            "lon": lon
        })

    # --- B. Normalization ---
    df_scores = pd.DataFrame(results).dropna()
    
    # Min-Max Normalize all 'raw' columns to 0-1
    for col in [c for c in df_scores.columns if c.startswith('raw_')]:
        min_v = df_scores[col].min()
        max_v = df_scores[col].max()
        # Create 'n_' column (Normalized)
        df_scores[f"n_{col.replace('raw_', '')}"] = (df_scores[col] - min_v) / (max_v - min_v)

    # --- C. Final Scoring (60/40 Split) ---
    
    # Profitability (40% Weight)
    # Factors: Low Price (50%), Low Load Stress (50%)
    df_scores['score_profit'] = (
        0.50 * (1 - df_scores['n_price']) +
        0.50 * (1 - df_scores['n_load'])
    )

    # Sustainability (60% Weight)
    # Factors: High Renewables (30%), Low Carbon (30%), High Stability (20%), Low Temp (20%)
    df_scores['score_sustain'] = (
        0.30 * df_scores['n_renew'] +
        0.30 * (1 - df_scores['n_carbon']) +
        0.20 * (1 - df_scores['n_volatility']) +
        0.20 * (1 - df_scores['n_temp'])
    )

    # Composite Score
    df_scores['gridcast_score'] = (
        0.40 * df_scores['score_profit'] + 
        0.60 * df_scores['score_sustain']
    )
    
    return df_scores.sort_values('gridcast_score', ascending=False)

# Run it
final_df = run_scoring_pipeline()
print("\n🏆 Top Recommended Datacenter Regions:")
print(final_df[['region', 'gridcast_score', 'score_sustain', 'score_profit']].head())

🚀 Starting GridCast Scoring for 13 regions...
  Analysing CAL...


/var/folders/h3/f3_68x516pj6d7h2119r5st00000gn/T/ipykernel_18746/3471390308.py:24: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end = datetime.utcnow()


  Analysing CAR...


/var/folders/h3/f3_68x516pj6d7h2119r5st00000gn/T/ipykernel_18746/3471390308.py:24: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end = datetime.utcnow()


  Analysing CENT...


...

  Analysing TEN...


/var/folders/h3/f3_68x516pj6d7h2119r5st00000gn/T/ipykernel_18746/3471390308.py:24: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end = datetime.utcnow()


  Analysing TEX...


/var/folders/h3/f3_68x516pj6d7h2119r5st00000gn/T/ipykernel_18746/3471390308.py:24: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end = datetime.utcnow()



🏆 Top Recommended Datacenter Regions:
   region  gridcast_score  score_sustain  score_profit
10     SW        1.000000       1.000000      1.000000
6      NE        0.871266       0.797489      0.981932
11    TEN        0.829193       0.748805      0.949775
1     CAR        0.807422       0.740955      0.907123
7      NY        0.796694       0.690959      0.955297
